In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse

# Replace with your folder path
base = Path(
    "/michorlab/yiyang/thesis/data/drug-perturbation/"
    "Srivatsan2020_sci-Plex"
)
src = base / "extracted"    # folder with raw files
out = base / "h5ad"         # folder where output will be put
out.mkdir(parents=True, exist_ok=True)

prefix = "GSM4150378_sciPlex3"

obs = pd.read_csv(src / f"{prefix}_pData.txt.gz", sep=" ")

var = pd.read_csv(
    src / f"{prefix}_A549_MCF7_K562_screen_gene.annotations.txt.gz",
    sep=" ",
    header=0,
    names=["gene_id", "gene_symbol"],
    dtype=str,
)

assert "cell" in obs.columns, "Unexpected metadata format; inspect obs.head()."
assert obs["cell"].notna().all()
assert obs["cell"].is_unique

obs.index = pd.Index(obs["cell"].astype(str), name="cell_id")
var.index = pd.Index(var["gene_id"], name="gene_index")

print(f"{len(obs):,} cells × {len(var):,} genes")
display(obs.head())
display(var.head())

799,317 cells × 110,983 genes


,cell,sample,Size_Factor,n.umi,hash_umis_W,pval_W,qval_W,top_to_second_best_ratio_W,top_oligo_W,hash_umis_P,...,vehicle,dose_pattern,dose_character,dose,treatment,pathway_level_1,pathway_level_2,product_name,target,pathway
cell_id,,,,,,,,,,,,,,,,,,,,,
A01_E09_RT_BC_100_Lig_BC_147,A01_E09_RT_BC_100_Lig_BC_147,Chem3_Screen2,1.751104,2957,79.0,0.000000e+00,0.000000e+00,5.665058,plate6_A9,194.0,...,False,1.0,10000.0,10000.0,S2718,Tyrosine kinase signaling,RTK activity,TAK-901,Aurora Kinase,Cell Cycle
A01_E09_RT_BC_100_Lig_BC_186,A01_E09_RT_BC_100_Lig_BC_186,Chem3_Screen2,0.904866,1528,117.0,0.000000e+00,0.000000e+00,69.651274,plate8_H3,111.0,...,False,4.0,10.0,10.0,S1143,Tyrosine kinase signaling,RTK activity,AG-490 (Tyrphostin B42),EGFR,Protein Tyrosine Kinase
A01_E09_RT_BC_100_Lig_BC_196,A01_E09_RT_BC_100_Lig_BC_196,Chem3_Screen2,1.113908,1881,197.0,0.000000e+00,0.000000e+00,64.333100,plate3_C2,253.0,...,False,2.0,1000.0,1000.0,S1090,Epigenetic regulation,Histone deacetylation,Abexinostat (PCI-24781),HDAC,Cytoskeletal Signaling
A01_E09_RT_BC_100_Lig_BC_213,A01_E09_RT_BC_100_Lig_BC_213,Chem3_Screen2,1.006722,1700,32.0,2.048502e-211,4.977859e-209,10.065182,plate9_E3,24.0,...,False,2.0,1000.0,1000.0,S1133,Cell cycle regulation,Aurora kinase activity,Alisertib (MLN8237),Aurora Kinase,Cell Cycle
A01_E09_RT_BC_100_Lig_BC_220,A01_E09_RT_BC_100_Lig_BC_220,Chem3_Screen2,0.846831,1430,327.0,0.000000e+00,0.000000e+00,126.355942,plate8_H10,446.0,...,False,1.0,10000.0,10000.0,S1692,DNA damage & DNA repair,Alkylating agent,Busulfan,DNA alkylator,DNA Damage


,gene_id,gene_symbol
gene_index,,
ENSG00000000003.14,ENSG00000000003.14,TSPAN6
ENSG00000000005.5,ENSG00000000005.5,TNMD
ENSG00000000419.12,ENSG00000000419.12,DPM1
ENSG00000000457.13,ENSG00000000457.13,SCYL3
ENSG00000000460.16,ENSG00000000460.16,C1orf112


In [3]:
matrix_file = (
    src / f"{prefix}_A549_MCF7_K562_screen_UMI.count.matrix.gz"
)

triplets = pd.read_csv(
    matrix_file,
    sep="\t",
    header=None,
    names=["gene", "cell", "count"],
    dtype=np.int64,
)


gene = triplets["gene"].to_numpy() - 1
cell = triplets["cell"].to_numpy() - 1
count = triplets["count"].to_numpy()

assert gene.min() >= 0 and gene.max() < len(var)
assert cell.min() >= 0 and cell.max() < len(obs)
assert (count >= 0).all()

X = sparse.csr_matrix(
    (count, (cell, gene)),
    shape=(len(obs), len(var)),
)

del triplets, gene, cell, count

X.eliminate_zeros()
X.sort_indices()

print("Matrix shape:", X.shape)
print("Nonzero entries:", X.nnz)

Reading and decompressing counts...
Building sparse matrix...
Matrix shape: (799317, 110983)
Nonzero entries: 1007419688


In [4]:
for frame in (obs, var):
    for col in frame.select_dtypes(include=["object", "string"]).columns:
        frame[col] = pd.Categorical(frame[col])

adata = ad.AnnData(X=X, obs=obs, var=var)
adata.var_names_make_unique()

adata.uns["dataset"] = "Srivatsan2020_sciPlex3"
adata.uns["expression"] = "Unnormalized UMI counts in X"
adata.uns["source_files"] = [
    matrix_file.name,
    f"{prefix}_pData.txt.gz",
    f"{prefix}_A549_MCF7_K562_screen_gene.annotations.txt.gz",
]

output_file = out / "sciplex3_all_cells.h5ad"
if output_file.exists():
    raise FileExistsError(f"Output already exists: {output_file}")

adata.write_h5ad(output_file, compression="gzip")

print(adata)
print("Saved:", output_file)

AnnData object with n_obs × n_vars = 799317 × 110983
    obs: 'cell', 'sample', 'Size_Factor', 'n.umi', 'hash_umis_W', 'pval_W', 'qval_W', 'top_to_second_best_ratio_W', 'top_oligo_W', 'hash_umis_P', 'pval_P', 'qval_P', 'top_to_second_best_ratio_P', 'top_oligo_P', 'rt_well', 'lig_well', 'pcr_well', 'pcr_plate', 'culture_plate', 'rt_plate', 'lig_plate', 'Combo', 'well_oligo', 'plate_oligo', 'cell_type', 'replicate', 'time_point', 'drug_dose', 'catalog_number', 'vehicle', 'dose_pattern', 'dose_character', 'dose', 'treatment', 'pathway_level_1', 'pathway_level_2', 'product_name', 'target', 'pathway'
    var: 'gene_id', 'gene_symbol'
    uns: 'dataset', 'expression', 'source_files'
Saved: /michorlab/yiyang/thesis/data/drug-perturbation/Srivatsan2020_sci-Plex/h5ad/sciplex3_all_cells.h5ad
